In [ ]:
import os, sys
import psycopg2

sys.path.append(os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))
from shared.embedding import embed
from _data_generator.pgvec_generator import generate


In [ ]:
conn = psycopg2.connect(
    host="pgvector_snoql_lab",
    database="vectordb",
    user="user",
    password="password"
)

cur = conn.cursor()

In [ ]:
DDL = """
-- EXTENSION
CREATE EXTENSION IF NOT EXISTS vector;

-- =========================
-- DROP (safe reset)
-- =========================
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS products;

-- =========================
-- PRODUCTS
-- =========================
CREATE TABLE products (
    id INT PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price NUMERIC(10,2),
    description TEXT,
    embedding VECTOR(384),
    created_at TIMESTAMP DEFAULT NOW()
);

-- =========================
-- SALES
-- =========================
CREATE TABLE sales (
    product_id INT REFERENCES products(id),
    sales_count INT,
    updated_at TIMESTAMP DEFAULT NOW(),
    PRIMARY KEY (product_id)
);

-- =========================
-- INDEXES
-- =========================
CREATE INDEX idx_products_embedding_cosine
ON products
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);

CREATE INDEX idx_products_category
ON products(category);
"""


def initialize_database():
    print("Connecting to database...")

    print("Initializing schema...")

    cur.execute(DDL)
    conn.commit()

    print("Database initialized successfully.")

    cur.execute("SELECT current_database();")
    print(cur.fetchone())

In [ ]:
initialize_database()

In [ ]:
generate()

In [ ]:
query = "electronics product"
query_vec = embed([query])[0].tolist()

cur.execute(
    """
    SELECT name, category, price
    FROM products
    WHERE category = 'electronics'
    -- ORDER BY embedding <=> %s::vector
    LIMIT 5
    """, (query_vec,)
)

results = cur.fetchall()

for r in results:
    print(">>>", r)

In [ ]:
# cur.execute("""
# DROP TABLE IF EXISTS sales;
# DROP TABLE IF EXISTS products;
# DROP EXTENSION IF EXISTS vector;
# """)
# conn.commit()
# print("Tables dropped")

In [ ]:
cur.close()
conn.close()